# Diabetes Readmission Prediction
## Data Cleaning, Feature Engineering, Encoding, and Modeling

## Import Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report


## Loading Data and Replacing Missing Values with Null

In [ ]:
df = pd.read_csv("data/train.csv")

# Remove ? with NULL
df = df.replace(r"^\s*\?\s*$", pd.NA, regex=True)
# Remove Unknown/Invalid with NULL
df = df.replace(r"^\s*Unknown/Invalid\s*$", pd.NA, regex=True)
# Remove empty values with NULL
df = df.replace(r"^\s*$", pd.NA, regex=True)

# From IDS_mapping.csv - setting unknown/invalid ID codes to NULL
ids_to_null = {
    "admission_type_id": [5, 6, 8],                # Not Available, NULL, Not Mapped
    "discharge_disposition_id": [18, 25, 26],      # NULL, Not Mapped, Unknown/Invalid
    "admission_source_id": [9, 15, 17, 20, 21],    # Not Available, NULL, Not Mapped, Unknown/Invalid
}

for col, bad_codes in ids_to_null.items():
    df[col] = df[col].replace(bad_codes, pd.NA)


## Verify Null Replacement

In [ ]:
# Check that mapped ID placeholders are gone
print((df["admission_type_id"].isin([5,6,8])).sum())
print((df["discharge_disposition_id"].isin([18,25,26])).sum())
print((df["admission_source_id"].isin([9,15,17,20,21])).sum())

# Check key placeholder strings are gone
for c in ["weight","payer_code","medical_specialty","race","gender","diag_1","diag_2","diag_3"]:
    print(c, (df[c].astype(str).str.strip() == "?").sum(), (df[c].astype(str).str.strip() == "Unknown/Invalid").sum())


## Check Feature Information

In [ ]:
df.shape
df.head()
df.info()
df.describe()


## Check Missing % and Unique Values

In [ ]:
# Print Missing % in each feature
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print("Missing % of data in features")
print(missing_pct.head(20))

# Print low-variance (few unique values)
nunique = df.nunique(dropna=False).sort_values()
print(nunique.head(20))


## EDA - Feature Analysis vs Readmission

In [ ]:
tmp = df.copy()
tmp["readmitted_bin"] = tmp["readmitted"].map({"No": 0, ">30": 1, "<30": 1})

features = [c for c in tmp.columns if c not in ["readmitted", "readmitted_bin"]]

for col in features:
    print("\n" + "="*60)
    print(f"Feature: {col}")

    if pd.api.types.is_numeric_dtype(tmp[col]):
        out = tmp.groupby("readmitted_bin")[col].agg(["count", "mean", "median", "std"])
        print(out)
    else:
        out = (
            tmp[[col, "readmitted_bin"]]
            .dropna()
            .groupby(col)["readmitted_bin"]
            .agg(["count", "mean"])
            .rename(columns={"mean": "readmission_rate"})
            .sort_values("readmission_rate", ascending=False)
        )
        print(out.head(15))


## Data Cleaning Pipeline

In [ ]:
# --- Step 1: Drop unnecessary columns ---
# NOTE: patient_nbr and encounter_id kept intentionally for history feature engineering below
df = df.drop(columns=[
    "payer_code",
    "id",
    "diag_1",
    "diag_2",
    "diag_3",
], errors="ignore")

# Dropping features with too many missing values
df = df.drop(columns=[
    "weight",
    "max_glu_serum",
    "A1Cresult",
    "medical_specialty",
], errors="ignore")

# Dropping features with no variation
df = df.drop(columns=[
    "troglitazone",
    "examide",
    "citoglipton",
], errors="ignore")

# Dropping near-zero variance medication columns
df = df.drop(columns=[
    "acetohexamide",
    "glipizide-metformin",
    "tolbutamide",
    "metformin-pioglitazone",
    "metformin-rosiglitazone",
    "glimepiride-pioglitazone",
    "tolazamide",
], errors="ignore")

# --- Step 2: Remove expired/hospice patients ---
# Convert to numeric first since column is stored as string
df["discharge_disposition_id"] = pd.to_numeric(df["discharge_disposition_id"], errors="coerce")
# From IDS_mapping.csv: 11=Expired, 13=Hospice/home, 14=Hospice/facility,
#                       19=Expired at home, 20=Expired in facility, 21=Expired unknown
before = len(df)
df = df[~df["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])]
print(f"Rows removed (expired/hospice): {before - len(df)}")

# --- Step 3: Engineer history feature (needs patient_nbr and encounter_id) ---
# Sort chronologically so cumcount() counts previous visits correctly
df = df.sort_values(["patient_nbr", "encounter_id"])
# previously_readmitted = 1 if this patient has been here before
df["previously_readmitted"] = (
    df.groupby("patient_nbr").cumcount() > 0
).astype(int)

# --- Step 4: Drop identifiers - no longer needed ---
df = df.drop(columns=["patient_nbr", "encounter_id"])

# --- Step 5: Handle missing values ---
# Drop tiny number of rows with missing gender
df = df.dropna(subset=["gender"])
# Fill remaining categoricals with 'Unknown'
for col in ["admission_type_id", "admission_source_id", "discharge_disposition_id", "race"]:
    df[col] = df[col].fillna("Unknown").astype(str)

# --- Verify ---
print(f"Rows remaining: {len(df)}")
print(f"Missing values remaining:\n{df.isna().sum()[df.isna().sum() > 0]}")
print(f"\npreviously_readmitted value counts:\n{df['previously_readmitted'].value_counts()}")


## Post-Cleaning Check

In [ ]:
# Print Missing % in each feature
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print("Missing % of data in features")
print(missing_pct.head(20))

# Print low-variance (few unique values)
nunique = df.nunique(dropna=False).sort_values()
print(nunique.head(20))


## Encoding

In [ ]:
# --- Ordinal encoding for age ---
# Age has a natural order so we encode as 0-9 rather than one-hot
age_order = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
             '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']
df['age'] = pd.Categorical(df['age'], categories=age_order, ordered=True).codes

# --- Binary encoding ---
df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
df['change'] = df['change'].map({'No': 0, 'Ch': 1})
df['diabetesMed'] = df['diabetesMed'].map({'No': 0, 'Yes': 1})

# --- Target encoding ---
# We just care about readmitted or not, not how long between admissions
df['readmitted'] = df['readmitted'].map({'No': 0, '>30': 1, '<30': 1})

# --- One-hot encoding for nominal categoricals ---
# drop_first=False: we don't imply race_A < race_B etc. in medical context
df = pd.get_dummies(df, columns=[
    'race',
    'admission_type_id',
    'admission_source_id',
    'discharge_disposition_id',
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'glipizide', 'glyburide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'miglitol', 'insulin',
    'glyburide-metformin'
], drop_first=False)

# --- Verify ---
print(f"Shape after encoding: {df.shape}")
print(f"Remaining non-numeric columns:\n{df.select_dtypes(include='object').columns.tolist()}")


## Feature Selection - Mutual Information

In [ ]:
from sklearn.feature_selection import mutual_info_classif

y = df["readmitted"]
X = df.drop(columns=["readmitted"])

# Calculate mutual information scores for all features
scores = pd.Series(
    mutual_info_classif(X, y, random_state=42),
    index=X.columns
).sort_values(ascending=False)

# Plot top 30 features
plt.figure(figsize=(12, 8))
plt.barh(scores.index[:30], scores.values[:30])
plt.xlabel("Mutual Information Score")
plt.title("Top 30 Features by Mutual Information Score")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

pd.set_option('display.max_rows', None)
print("\nAll feature scores:")
print(scores)
pd.reset_option('display.max_rows')


## Define Feature Sets

In [ ]:
# 21 curated features - removing _No medication columns (likely noise)
selected_features = [
    # Strong numeric features
    'number_inpatient', 'previously_readmitted', 'number_emergency',
    'number_diagnoses', 'num_medications', 'number_outpatient', 'age',
    # Patient info
    'gender', 'race_Caucasian',
    # Admission info
    'discharge_disposition_id_1.0', 'admission_source_id_7',
    'admission_source_id_1', 'admission_source_id_4', 'admission_type_id_2',
    # Medication activity (actual usage, not absence)
    'insulin_Up', 'insulin_Steady', 'insulin_No',
    'glipizide_Steady', 'miglitol_Steady',
    # Other meaningful features
    'diabetesMed', 'change',
]

# 54 features - threshold > 0.001 mutual information
selected_features_54 = scores[scores > 0.001].index.tolist()

# 21 feature X/y
X = df[selected_features]
y = df["readmitted"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 54 feature X/y
X2 = df[selected_features_54]
y2 = df["readmitted"]
X2_train, X2_val, y2_train, y2_val = train_test_split(X2, y2, test_size=0.2, random_state=42)

print(f"21 features - X shape: {X.shape}")
print(f"54 features - X2 shape: {X2.shape}")
print(f"y value counts:\n{y.value_counts()}")


## Model 1: Logistic Regression (21 features)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:, 1]

print(f"Logistic Regression (21 features):")
print(f"Accuracy:  {accuracy_score(y_val, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, y_prob):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_val, y_pred))


## Logistic Regression Coefficients

In [ ]:
coefficients = pd.Series(
    model.coef_[0],
    index=selected_features
).sort_values(ascending=False)

print("Feature coefficients (positive = increases readmission risk):")
print(coefficients)

plt.figure(figsize=(10, 7))
colors = ['red' if c > 0 else 'blue' for c in coefficients.values]
plt.barh(coefficients.index, coefficients.values, color=colors)
plt.xlabel("Coefficient Value")
plt.title("Logistic Regression Coefficients\nRed = increases readmission risk, Blue = decreases risk")
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## Model 2: Random Forest (21 features)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_val)
rf_prob = rf_model.predict_proba(X_val)[:, 1]

print(f"Random Forest (21 features):")
print(f"Accuracy:  {accuracy_score(y_val, rf_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, rf_prob):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_val, rf_pred))


## Model 3: Random Forest (54 features)

In [ ]:
rf_model2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model2.fit(X2_train, y2_train)

rf2_pred = rf_model2.predict(X2_val)
rf2_prob = rf_model2.predict_proba(X2_val)[:, 1]

print(f"Random Forest (54 features):")
print(f"Accuracy:  {accuracy_score(y2_val, rf2_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y2_val, rf2_prob):.4f}")
print(f"\nClassification Report:")
print(classification_report(y2_val, rf2_pred))


## Model 4: KNN (21 features)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Scale features - critical for KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn.fit(X_train_scaled, y_train)

knn_pred = knn.predict(X_val_scaled)
knn_prob = knn.predict_proba(X_val_scaled)[:, 1]

print(f"KNN (21 features):")
print(f"Accuracy:  {accuracy_score(y_val, knn_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, knn_prob):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_val, knn_pred))


## Model 5: Decision Tree (21 features)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42, max_depth=5)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_val)
dt_prob = dt.predict_proba(X_val)[:, 1]

print(f"Decision Tree (21 features):")
print(f"Accuracy:  {accuracy_score(y_val, dt_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, dt_prob):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_val, dt_pred))


## Visualize Decision Tree

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 10))
plot_tree(dt,
          feature_names=selected_features,
          class_names=["Not Readmitted", "Readmitted"],
          filled=True,
          rounded=True,
          fontsize=10)
plt.title("Decision Tree (max_depth=5)")
plt.tight_layout()
plt.show()


## Tuning: Random Forest (54 features)

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_leaf': [1, 5, 10],
    'max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X2_train, y2_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_
best_rf_pred = best_rf.predict(X2_val)
best_rf_prob = best_rf.predict_proba(X2_val)[:, 1]

print(f"\nTuned Random Forest Results:")
print(f"Accuracy:  {accuracy_score(y2_val, best_rf_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y2_val, best_rf_prob):.4f}")
print(f"\nClassification Report:")
print(classification_report(y2_val, best_rf_pred))


## Tuning: Decision Tree (21 features)

In [ ]:
param_grid_dt = {
    'max_depth': [3, 5, 10, 20, None],
    'min_samples_leaf': [1, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid_dt,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

dt_grid.fit(X_train, y_train)

print(f"Best parameters: {dt_grid.best_params_}")
print(f"Best CV ROC-AUC: {dt_grid.best_score_:.4f}")

best_dt = dt_grid.best_estimator_
best_dt_pred = best_dt.predict(X_val)
best_dt_prob = best_dt.predict_proba(X_val)[:, 1]

print(f"\nTuned Decision Tree Results:")
print(f"Accuracy:  {accuracy_score(y_val, best_dt_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, best_dt_prob):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_val, best_dt_pred))


## Tuning: Logistic Regression (21 features)

In [ ]:
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga']
}

lr_grid = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=5000),
    param_grid_lr,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

lr_grid.fit(X_train, y_train)

print(f"Best parameters: {lr_grid.best_params_}")
print(f"Best CV ROC-AUC: {lr_grid.best_score_:.4f}")

best_lr = lr_grid.best_estimator_
best_lr_pred = best_lr.predict(X_val)
best_lr_prob = best_lr.predict_proba(X_val)[:, 1]

print(f"\nTuned Logistic Regression Results:")
print(f"Accuracy:  {accuracy_score(y_val, best_lr_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_val, best_lr_prob):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_val, best_lr_pred))


## Final Model Comparison

In [ ]:
print(f"{'Model':<35} {'Accuracy':>10} {'ROC-AUC':>10}")
print("-" * 57)
print(f"{'Tuned Random Forest (54f)':<35} {accuracy_score(y2_val, best_rf_pred):>10.4f} {roc_auc_score(y2_val, best_rf_prob):>10.4f}")
print(f"{'Tuned Logistic Regression (21f)':<35} {accuracy_score(y_val, best_lr_pred):>10.4f} {roc_auc_score(y_val, best_lr_prob):>10.4f}")
print(f"{'Tuned Decision Tree (21f)':<35} {accuracy_score(y_val, best_dt_pred):>10.4f} {roc_auc_score(y_val, best_dt_prob):>10.4f}")
print(f"{'Random Forest (54f)':<35} {accuracy_score(y2_val, rf2_pred):>10.4f} {roc_auc_score(y2_val, rf2_prob):>10.4f}")
print(f"{'Logistic Regression (21f)':<35} {accuracy_score(y_val, y_pred):>10.4f} {roc_auc_score(y_val, y_prob):>10.4f}")
print(f"{'Random Forest (21f)':<35} {accuracy_score(y_val, rf_pred):>10.4f} {roc_auc_score(y_val, rf_prob):>10.4f}")
print(f"{'Decision Tree (21f)':<35} {accuracy_score(y_val, dt_pred):>10.4f} {roc_auc_score(y_val, dt_prob):>10.4f}")
print(f"{'KNN (21f)':<35} {accuracy_score(y_val, knn_pred):>10.4f} {roc_auc_score(y_val, knn_prob):>10.4f}")
